In [3]:
# 02_feature_engineering.ipynb

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

# ── 1. Load all 6 months ──────────────────────────────────────────────────────
months = ["2025-11", "2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]

price_frames = []
for month in months:
    url = f"https://storage.data.gov.my/pricecatcher/pricecatcher_{month}.parquet"
    temp = pd.read_parquet(url)
    temp["date"] = pd.to_datetime(temp["date"])
    temp["month"] = temp["date"].dt.to_period("M").astype(str)
    price_frames.append(temp)
    print(f"Loaded {month}: {temp.shape}")

price_all = pd.concat(price_frames, ignore_index=True)
print("Combined shape:", price_all.shape)

Loaded 2025-11: (1783107, 5)
Loaded 2025-12: (1782274, 5)
Loaded 2026-01: (1976483, 5)
Loaded 2026-02: (1369552, 5)
Loaded 2026-03: (1547695, 5)
Loaded 2026-04: (1745608, 5)
Combined shape: (10204719, 5)


In [4]:
# ── 2. Load lookups and clean ─────────────────────────────────────────────────
item   = pd.read_parquet("https://storage.data.gov.my/pricecatcher/lookup_item.parquet")
premise = pd.read_parquet("https://storage.data.gov.my/pricecatcher/lookup_premise.parquet")

item_clean   = item.dropna(subset=["item_code","item","unit","item_group","item_category"]).copy()
premise_clean = premise.dropna(subset=["premise_code","premise","premise_type","state","district"]).copy()

# ── 3. Join ───────────────────────────────────────────────────────────────────
df = (
    price_all
    .merge(item_clean,    on="item_code",    how="left")
    .merge(premise_clean, on="premise_code", how="left")
)
print("Joined shape:", df.shape)

Joined shape: (10204719, 14)


In [5]:
# ── 4. Price cleaning (same cutoffs as notebook 01) ───────────────────────────
df = df[df["price"] > 0].copy()

# IMPORTANT: compute cutoffs on full 6-month dataset, not per-month
lower_cutoff = df["price"].quantile(0.01)
upper_cutoff = df["price"].quantile(0.99)
print(f"Price cutoffs: {lower_cutoff:.2f} – {upper_cutoff:.2f}")

df_clean = df[(df["price"] >= lower_cutoff) & (df["price"] <= upper_cutoff)].copy()
print("Cleaned rows:", df_clean.shape[0])

Price cutoffs: 1.60 – 49.90
Cleaned rows: 10008188


In [6]:
# ── 5. Text cleaning ──────────────────────────────────────────────────────────
text_cols = ["item","unit","item_group","item_category","premise","premise_type","state","district"]
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

In [7]:
# ── 6. Basket filter (same 9 categories) ─────────────────────────────────────
BASKET_CATEGORIES = [
    "AYAM", "TELUR", "BERAS", "MINYAK DAN LEMAK",
    "BAWANG", "SAYUR-SAYURAN", "BAHAN LAUT", "IKAN DARAT", "BUAH-BUAHAN"
]
df_basket = df_clean[df_clean["item_category"].isin(BASKET_CATEGORIES)].copy()
print("Basket rows:", df_basket.shape[0])
print("Months covered:", df_basket["month"].unique())

Basket rows: 6677619
Months covered: <ArrowStringArray>
['2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']
Length: 6, dtype: str


In [8]:
# ── 7. Feature engineering ────────────────────────────────────────────────────

# 7a. Time features (cyclical encoding for month)
df_basket["year"]       = df_basket["date"].dt.year
df_basket["month_num"]  = df_basket["date"].dt.month
df_basket["month_sin"]  = np.sin(2 * np.pi * df_basket["month_num"] / 12)
df_basket["month_cos"]  = np.cos(2 * np.pi * df_basket["month_num"] / 12)

# 7b. Rolling item price signal: mean price of this item in the PREVIOUS month
#     This gives the model a "recent price trend" feature without leaking future data
item_monthly_avg = (
    df_basket
    .groupby(["item_code", "month"])["price"]
    .mean()
    .reset_index()
    .rename(columns={"price": "item_monthly_avg_price"})
)
# Shift: sort by month and lag by 1
item_monthly_avg = item_monthly_avg.sort_values(["item_code", "month"])
item_monthly_avg["item_prev_month_price"] = (
    item_monthly_avg.groupby("item_code")["item_monthly_avg_price"].shift(1)
)
df_basket = df_basket.merge(item_monthly_avg[["item_code","month","item_prev_month_price"]],
                             on=["item_code","month"], how="left")

# 7c. Price volatility per item (std over the full 6-month window)
item_volatility = (
    df_basket.groupby("item_code")["price"]
    .std()
    .reset_index()
    .rename(columns={"price": "item_price_volatility"})
)
df_basket = df_basket.merge(item_volatility, on="item_code", how="left")

# 7d. Premise-level price index (how expensive is this premise on average)
premise_avg = (
    df_basket.groupby("premise_code")["price"]
    .mean()
    .reset_index()
    .rename(columns={"price": "premise_price_index"})
)
df_basket = df_basket.merge(premise_avg, on="premise_code", how="left")

print(df_basket[["month_sin","month_cos","item_prev_month_price",
                  "item_price_volatility","premise_price_index"]].describe())

          month_sin     month_cos  item_prev_month_price  \
count  6.677619e+06  6.677619e+06           5.493709e+06   
mean   4.207141e-01  4.754984e-01           1.198163e+01   
std    5.420410e-01  5.505384e-01           8.246426e+00   
min   -5.000000e-01 -5.000000e-01           2.279452e+00   
25%   -2.449294e-16  6.123234e-17           6.463105e+00   
50%    5.000000e-01  8.660254e-01           1.002532e+01   
75%    8.660254e-01  8.660254e-01           1.447925e+01   
max    1.000000e+00  1.000000e+00           4.732340e+01   

       item_price_volatility  premise_price_index  
count           6.677619e+06         6.677619e+06  
mean            2.856221e+00         1.186410e+01  
std             1.753761e+00         1.597707e+00  
min             1.378405e-01         6.438202e+00  
25%             1.398129e+00         1.086167e+01  
50%             2.496024e+00         1.164679e+01  
75%             3.467220e+00         1.268360e+01  
max             9.369086e+00         2.8500

In [9]:
# ── 8. Build the MODEL DATASET ────────────────────────────────────────────────
# Aggregate to (month, state, premise_type) — same grain as your monthly_basket
# This is what the model will predict: average basket price for this combination

model_df = (
    df_basket
    .groupby(["month", "year", "month_num", "state", "premise_type"], as_index=False)
    .agg(
        basket_price           = ("price",                 "mean"),
        median_basket_price    = ("price",                 "median"),
        price_std              = ("price",                 "std"),
        num_records            = ("price",                 "count"),
        num_items              = ("item_code",             "nunique"),
        num_premises           = ("premise_code",          "nunique"),
        avg_item_volatility    = ("item_price_volatility", "mean"),
        avg_premise_index      = ("premise_price_index",   "mean"),
        avg_prev_month_price   = ("item_prev_month_price", "mean"),
        month_sin              = ("month_sin",             "first"),
        month_cos              = ("month_cos",             "first"),
    )
    .sort_values(["state", "premise_type", "month"])
)

print("Model dataset shape:", model_df.shape)
print(model_df.head(20))

Model dataset shape: (478, 16)
       month  year  month_num  state  premise_type  basket_price  \
0    2025-11  2025         11  Johor   Hypermarket     11.436141   
80   2025-12  2025         12  Johor   Hypermarket     12.363756   
160  2026-01  2026          1  Johor   Hypermarket     12.640538   
240  2026-02  2026          2  Johor   Hypermarket     11.918766   
320  2026-03  2026          3  Johor   Hypermarket     11.621693   
399  2026-04  2026          4  Johor   Hypermarket     11.067907   
1    2025-11  2025         11  Johor  Kedai Runcit     12.630215   
81   2025-12  2025         12  Johor  Kedai Runcit     13.252870   
161  2026-01  2026          1  Johor  Kedai Runcit     13.556277   
241  2026-02  2026          2  Johor  Kedai Runcit     13.174604   
321  2026-03  2026          3  Johor  Kedai Runcit     12.584874   
400  2026-04  2026          4  Johor  Kedai Runcit     12.635980   
2    2025-11  2025         11  Johor   Pasar Basah     12.097406   
82   2025-12  202

In [10]:
# ── 9. Encode categoricals ────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

le_state         = LabelEncoder()
le_premise_type  = LabelEncoder()

model_df["state_encoded"]        = le_state.fit_transform(model_df["state"])
model_df["premise_type_encoded"] = le_premise_type.fit_transform(model_df["premise_type"])

print("State classes:", le_state.classes_)
print("Premise type classes:", le_premise_type.classes_)

State classes: ['Johor' 'Kedah' 'Kelantan' 'Melaka' 'Negeri Sembilan' 'Pahang' 'Perak'
 'Perlis' 'Pulau Pinang' 'Sabah' 'Sarawak' 'Selangor' 'Terengganu'
 'W.P. Kuala Lumpur' 'W.P. Labuan' 'W.P. Putrajaya']
Premise type classes: ['Borong' 'Hypermarket' 'Kedai Runcit' 'Pasar Basah' 'Pasar Mini'
 'Pasar Raya / Supermarket']


In [11]:
# ── 10. Save outputs ──────────────────────────────────────────────────────────
model_df.to_parquet("../data/processed/model_df.parquet", index=False)
df_basket.to_parquet("../data/processed/df_basket_6mo.parquet", index=False)
print("Saved.")

Saved.
